# 04 — GNN Training & Evaluation

**Goal:** Train a GAT-based GNN (`DTNetGNN`) to predict disruption propagation across the supply-chain network, compare it against an `IsolatedBaseline` MLP that ignores graph structure, and extract which connections the model learns to prioritise.

**Thesis relevance:**
- RQ1/RQ2: Does the GNN predict cascade severity better than a node-local model?
- RQ3: Which supply-chain edges drive disruption propagation (attention weights)?

**Outputs saved to `results/`:** training curves, attention graph, scenario scatter plots.

In [ ]:
%matplotlib inline
import os, sys, pickle, warnings
from pathlib import Path
from datetime import date

import numpy as np
import torch
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import networkx as nx
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# ── Project root on path (works locally and on Colab)
ROOT = Path('..').resolve()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.gnn.dataset import build_dataloaders
from src.gnn.model import DTNetGNN, IsolatedBaseline
from src.gnn.train import run_training, GNN_SAVE_PATH, BASELINE_SAVE_PATH
from src.gnn.evaluate import (
    run_evaluation, _extract_attention,
    _find_scenario_run, _eval_scenario,
    LAYER_ORDER, ONE_HOT_START,
)

# ── Light theme
BG = 'white'
matplotlib.rcParams.update({
    'figure.facecolor': BG,  'axes.facecolor': BG,
    'axes.edgecolor':  '#CCCCCC', 'axes.labelcolor': '#333333',
    'xtick.color': '#333333',  'ytick.color': '#333333', 'text.color': '#333333',
    'grid.color':  '#EEEEEE', 'grid.alpha': 0.8,
    'legend.facecolor': '#F8F8F8', 'legend.edgecolor': '#CCCCCC',
    'figure.dpi': 100,
})

# ── Layer colours (CODING_PATTERNS.md)
LAYER_COLORS = {
    'supplier':     '#4c7aff',
    'logistics':    '#9b59b6',
    'plant':        '#2ecc71',
    'machine':      '#f39c12',
    'distribution': '#e74c3c',
}

DATE        = date.today().strftime('%Y%m%d')
PKL_PATH    = ROOT / 'data' / 'processed' / 'simulation_runs.pkl'
RESULTS     = Path('results')
RESULTS.mkdir(parents=True, exist_ok=True)
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device : {DEVICE}')
print(f'PKL    : {PKL_PATH}  (exists={PKL_PATH.exists()})')
print(f'Results: {RESULTS.resolve()}')

## 1. Dataset & Statistics

Load `simulation_runs.pkl` and print a summary.  Each run has:
- `initial_features` — 10-dim node state before disruption injection
- `final_severities` — per-node disruption severity after 10 simulation steps
- `initial_disruption` — which nodes were seeded and at what severity
- `edge_index`, `node_order` — shared topology

In [ ]:
from src.data.loader import load_csv
from src.data.preprocess import preprocess
from src.data.entity_mapping import build_entity_mappings
from src.graph.topology import infer_topology
from src.graph.builder import build_graph

assert PKL_PATH.exists(), f'PKL not found: {PKL_PATH}. Run 03_simulation_runs.ipynb first.'

with open(PKL_PATH, 'rb') as fh:
    runs = pickle.load(fh)

# ── Build graph (same as notebook 02) ─────────────────────────────────────
CSV_FILENAME = 'updated_data.csv'
df_raw       = load_csv(CSV_FILENAME)
df_clean, _  = preprocess(df_raw)
em           = build_entity_mappings(df_raw)
nodes, edges = infer_topology(em)
G            = build_graph(nodes, edges, df_clean)
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

train_loader, val_loader, test_loader = build_dataloaders(PKL_PATH, G=G)

n_nodes = len(runs[0]['node_order'])
n_feats = len(runs[0]['initial_features'][0])
n_edges = len(runs[0]['edge_index'])
all_sev = np.concatenate([r['final_severities'] for r in runs])
single_n = sum(1 for r in runs if len(r['initial_disruption']) == 1)
multi_n  = len(runs) - single_n

print(f'Total runs      : {len(runs):,}')
print(f'Nodes / run     : {n_nodes}   Features: {n_feats}   Edges: {n_edges}')
print(f'Single-node dis : {single_n:,}  ({100*single_n/len(runs):.1f}%)')
print(f'Multi-node  dis : {multi_n:,}  ({100*multi_n/len(runs):.1f}%)')
print(f'Mean severity   : {all_sev.mean():.4f}   Max: {all_sev.max():.4f}')
print(f'Disrupted frac  : {np.mean(all_sev > 0)*100:.1f}%  (nodes with final sev > 0)')
print()

# ── Layer distribution
print('Node layer distribution (from first run):')
raw_f0    = np.array(runs[0]['initial_features'])
layer_ids = np.argmax(raw_f0[:, 5:10], axis=1)
for li, name in enumerate(LAYER_ORDER):
    cnt = int((layer_ids == li).sum())
    print(f'  {name:<14} {cnt:3d} nodes')

## 2. Train Both Models

Train `DTNetGNN` (GAT, uses graph structure) and `IsolatedBaseline` (MLP, no graph).

If checkpoints already exist in `results/`, training is skipped and the saved models are loaded instead.

In [ ]:
GNN_PT  = Path(GNN_SAVE_PATH)
BASE_PT = Path(BASELINE_SAVE_PATH)

if GNN_PT.exists() and BASE_PT.exists():
    print('Checkpoints found — skipping training.')
    print(f'  GNN      : {GNN_PT}')
    print(f'  Baseline : {BASE_PT}')
    gnn_history = baseline_history = None
else:
    print('Training both models (this may take several minutes on CPU)...')
    gnn_history, baseline_history = run_training(
        train_loader, val_loader, device=DEVICE
    )
    best_g = gnn_history['best_val_loss']
    best_b = baseline_history['best_val_loss']
    ep_g   = gnn_history['best_epoch']
    ep_b   = baseline_history['best_epoch']
    print(f'DTNetGNN      best val MSE = {best_g:.6f}  @ epoch {ep_g}')
    print(f'IsolatedBase  best val MSE = {best_b:.6f}  @ epoch {ep_b}')

## 3. Training & Validation Loss Curves

Shows how both models converge.  The early-stopping checkpoint is marked with a white dot.

> If training was skipped (checkpoints loaded), re-run the training cell first.

In [ ]:
if not gnn_history:
    print('No training history in memory — re-run the training cell to generate curves.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)

    for ax, hist, title, tc, vc in [
        (axes[0], gnn_history,      'DTNetGNN',         '#4c7aff', '#f39c12'),
        (axes[1], baseline_history, 'IsolatedBaseline', '#9b59b6', '#2ecc71'),
    ]:
        ep   = range(len(hist['train_loss']))
        best = hist['best_epoch']
        bval = hist['best_val_loss']

        ax.plot(ep, hist['train_loss'], color=tc, lw=1.8, label='Train MSE', alpha=0.9)
        ax.plot(ep, hist['val_loss'],   color=vc, lw=1.8, ls='--', label='Val MSE', alpha=0.9)
        ax.axvline(best, color='#AAAAAA', lw=1.0, ls=':', alpha=0.6,
                   label=f'Best checkpoint (ep {best})')
        ax.scatter([best], [bval], color='#333333', s=70, zorder=6)
        ax.annotate(f'{bval:.5f}', xy=(best, bval),
                    xytext=(best + max(len(hist["train_loss"]) // 12, 1), bval * 1.08),
                    color='#333333', fontsize=8,
                    arrowprops=dict(arrowstyle='->', color='#333333', lw=0.7))

        ax.set_title(title, fontsize=13, pad=10)
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('MSE Loss', fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True)
        ax.set_facecolor(BG)

    fig.suptitle('Training & Validation Loss — DTNetGNN vs IsolatedBaseline',
                 fontsize=14, color='#333333', y=1.02)
    plt.tight_layout()
    p = RESULTS / f'dtnet_training_curves_{DATE}.png'
    fig.savefig(p, dpi=200, bbox_inches='tight', facecolor=BG)
    print(f'Saved: {p}')
    plt.show()

## 4. Test-Set Evaluation

Load the best checkpoints, evaluate on the held-out test split (15%), and print:
- Overall MSE / MAE / R² for both models
- Per-node-type breakdown (supplier, logistics, plant, machine, distribution)
- Top-10 most-attended edges from the GAT layers

In [ ]:
eval_results = run_evaluation(test_loader, runs=runs, device=DEVICE)

# ── Overall comparison DataFrame
gm = eval_results['gnn_test']
bm = eval_results['baseline_test']

df_overall = pd.DataFrame([
    {'Model': 'DTNetGNN',         'MSE': gm['mse'], 'MAE': gm['mae'], 'R2': gm['r2']},
    {'Model': 'IsolatedBaseline', 'MSE': bm['mse'], 'MAE': bm['mae'], 'R2': bm['r2']},
]).set_index('Model').round(6)

print('── Overall Test-Set Performance ──────────────────────────')
print(df_overall.to_string())

# ── Per-type DataFrame
rows = []
for layer in LAYER_ORDER:
    for mname, key in [('GNN', 'per_type_gnn'), ('Baseline', 'per_type_base')]:
        m = eval_results[key].get(layer)
        if m:
            rows.append({'Layer': layer, 'Model': mname,
                         'MSE': m['mse'], 'MAE': m['mae'], 'R2': m['r2']})
df_pt = pd.DataFrame(rows).set_index(['Layer', 'Model']).round(6)
print()
print('── Per-Node-Type Performance ─────────────────────────────')
print(df_pt.to_string())

# ── Keep models in memory for attention + scenario cells
in_ch = next(iter(test_loader)).x.shape[1]
gnn_model = DTNetGNN(in_channels=in_ch).to(DEVICE)
gnn_model.load_state_dict(torch.load(GNN_SAVE_PATH, map_location=DEVICE))
gnn_model.eval()
base_model = IsolatedBaseline(in_channels=in_ch).to(DEVICE)
base_model.load_state_dict(torch.load(BASELINE_SAVE_PATH, map_location=DEVICE))
base_model.eval()
print()
print('Models loaded into gnn_model / base_model for downstream cells.')

## 5. Attention Weight Visualization

**Key thesis result.** The GAT layers learn to assign different attention scores to each supply-chain edge.  High-attention edges are the connections the model has determined are most predictive of disruption propagation.

- **Bright/thick golden edges** = high attention = critical dependency
- **Dark/thin edges** = low attention = less important for cascade prediction
- Nodes coloured by supply-chain layer (supplier=blue … distribution=red)

This answers: *"Which network connections does the GNN learn to monitor?"*

In [ ]:
# ── Extract attention from the first test graph
first_data = next(iter(test_loader)).to_data_list()[0]
attn       = _extract_attention(gnn_model, first_data, DEVICE)

# ── Recover node-layer info from raw initial_features (pre-normalisation)
run0       = runs[0]
node_order = run0['node_order']
edge_pairs = run0['edge_index']   # list of [src_idx, dst_idx]
raw_feats  = np.array(run0['initial_features'])
layer_ids  = np.argmax(raw_feats[:, 5:10], axis=1)
node_layer = {node_order[i]: LAYER_ORDER[int(layer_ids[i])] for i in range(len(node_order))}

# ── Attention lookup: (src_idx, dst_idx) -> attention weight
top_k_dict = {(e['src'], e['dst']): e['attention'] for e in attn['top_k_edges']}
max_attn   = max(top_k_dict.values()) if top_k_dict else 1.0

def _edge_color(val):
    t = val / max_attn
    r = int(0xCC + (0xFF - 0xCC) * t)
    g = int(0xCC + (0xD7 - 0xCC) * (1.0 - t))
    b = int(0xDD * (1.0 - t))
    return f'#{r:02x}{g:02x}{b:02x}'

# ── Hierarchical positions: supplier (top) → distribution (bottom)
y_map = {'supplier': 4, 'logistics': 3, 'plant': 2, 'machine': 1, 'distribution': 0}
layer_bucket = {l: [] for l in LAYER_ORDER}
for nid in node_order:
    layer_bucket[node_layer[nid]].append(nid)

max_n   = max(len(v) for v in layer_bucket.values())
x_scale = max(1.8, 12.0 / max_n)
pos = {}
for layer, nlist in layer_bucket.items():
    n = len(nlist)
    for k, nid in enumerate(nlist):
        pos[nid] = ((k - (n - 1) / 2.0) * x_scale, y_map[layer])

# ── Build NetworkX graph
VizG = nx.DiGraph()
VizG.add_nodes_from(node_order)
for ep in edge_pairs:
    VizG.add_edge(node_order[ep[0]], node_order[ep[1]])

edge_list   = [(node_order[ep[0]], node_order[ep[1]]) for ep in edge_pairs]
edge_widths = [0.4 + 4.5 * top_k_dict.get((ep[0], ep[1]), 0.0) / max_attn
               for ep in edge_pairs]
edge_colors = [_edge_color(top_k_dict.get((ep[0], ep[1]), 0.0))
               for ep in edge_pairs]

fig, ax = plt.subplots(figsize=(18, 9), facecolor=BG)
ax.set_facecolor(BG)

nx.draw_networkx_edges(
    VizG, pos, edgelist=edge_list, width=edge_widths,
    edge_color=edge_colors, arrows=True, arrowsize=12,
    ax=ax, alpha=0.82, connectionstyle='arc3,rad=0.08',
)
nx.draw_networkx_nodes(
    VizG, pos,
    node_color=[LAYER_COLORS[node_layer[n]] for n in VizG.nodes()],
    node_size=420, ax=ax, linewidths=1.2, edgecolors='#555555',
)
nx.draw_networkx_labels(VizG, pos, ax=ax, font_size=5.5, font_color='#333333')

# Layer labels along the left margin
for layer, y in y_map.items():
    ax.text(-x_scale * max_n / 2 - 1.8, y, layer.capitalize(),
            va='center', ha='right',
            color=LAYER_COLORS[layer], fontsize=10, fontweight='bold')

# Legend
patches = [mpatches.Patch(color=LAYER_COLORS[l], label=l.capitalize())
           for l in LAYER_ORDER]
patches += [
    mpatches.Patch(color='#ffd700', label='High attention (critical link)'),
    mpatches.Patch(color='#CCCCDD', label='Low attention'),
]
ax.legend(handles=patches, loc='upper right', fontsize=9, framealpha=0.88)
ax.set_title(
    'GAT Attention Weights — Which Supply-Chain Connections Matter Most?',
    color='#333333', fontsize=14, pad=15,
)
ax.axis('off')
plt.tight_layout()
p = RESULTS / f'dtnet_attention_graph_{DATE}.png'
fig.savefig(p, dpi=200, bbox_inches='tight', facecolor=BG)
print(f'Saved: {p}')
print(f'\nTop-{len(attn["top_k_edges"])} most-attended edges:')
for e in attn['top_k_edges']:
    src_lbl = e.get('src_name', f'node_{e["src"]}')
    dst_lbl = e.get('dst_name', f'node_{e["dst"]}')
    print(f'  #{e["rank"]:2d}  {src_lbl:<28} -> {dst_lbl:<28}  attn={e["attention"]:.4f}')
plt.show()

## 6. Scenario Analysis — Predicted vs Actual Severity

Three representative disruption scenarios from the dataset:
1. **Single Supplier Failure** — upstream localised shock
2. **Multi-Node Failure** — simultaneous disruption of 2+ nodes
3. **High-Cascade Scenario** — the run with the largest observed cascade spread

Each scatter plot shows **one point per node** coloured by layer type:
- **Filled circles (●)** = GNN prediction
- **X markers (✕)** = Baseline prediction
- **White dashed line** = perfect prediction

Points close to the diagonal = accurate prediction.  Points above the diagonal = over-prediction.

In [ ]:
# ── Scaler fitted on all runs (qualitative scenario display — not a benchmark)
all_feats = np.vstack([np.array(r['initial_features'], dtype=np.float32) for r in runs])
scaler    = StandardScaler().fit(all_feats)
scaler.scale_ = np.where(scaler.scale_ == 0.0, 1.0, scaler.scale_)

# ── Select 3 representative scenarios
sc1 = _find_scenario_run(runs, 'single_supplier')
sc2 = _find_scenario_run(runs, 'multi_node')
sc3 = max(runs, key=lambda r: sum(s > 0.3 for s in r['final_severities']))

scenarios = [
    (sc1, 'Single Supplier Failure'),
    (sc2, 'Multi-Node Failure'),
    (sc3, 'High-Cascade Scenario'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)

for ax, (run_dict, label) in zip(axes, scenarios):
    if run_dict is None:
        ax.text(0.5, 0.5, 'No matching run found',
                transform=ax.transAxes, ha='center', color='#333333', fontsize=11)
        ax.set_title(label, color='#333333', fontsize=11)
        ax.set_facecolor(BG)
        continue

    res    = _eval_scenario(gnn_model, base_model, run_dict, scaler, DEVICE)
    y_true = np.array(res['y_true'])
    gnn_p  = np.array(res['gnn_pred'])
    base_p = np.array(res['baseline_pred'])

    # Node types from raw (pre-normalisation) features
    raw_f = np.array(run_dict['initial_features'])
    li    = np.argmax(raw_f[:, 5:10], axis=1)

    for l_idx, layer in enumerate(LAYER_ORDER):
        mask = li == l_idx
        if not mask.any():
            continue
        c = LAYER_COLORS[layer]
        ax.scatter(y_true[mask], gnn_p[mask],  color=c, s=60, alpha=0.85,
                   marker='o', zorder=3)
        ax.scatter(y_true[mask], base_p[mask], color=c, s=45, alpha=0.40,
                   marker='X', zorder=2)

    # Perfect-prediction diagonal
    lim = float(max(y_true.max(), gnn_p.max(), base_p.max())) + 0.06
    ax.plot([0, lim], [0, lim], '--', color='#AAAAAA', lw=1.2, alpha=0.7)

    # Metric annotations
    gm2  = res['gnn_metrics']
    bm2  = res['baseline_metrics']
    bbox = dict(boxstyle='round,pad=0.25', facecolor='#EAF0FA', alpha=0.88)
    ax.text(0.04, 0.97, f'GNN   R2={gm2["r2"]:.3f}  MAE={gm2["mae"]:.3f}',
            transform=ax.transAxes, va='top', fontsize=8.5,
            color='#2255BB', bbox=bbox)
    ax.text(0.04, 0.88, f'Base  R2={bm2["r2"]:.3f}  MAE={bm2["mae"]:.3f}',
            transform=ax.transAxes, va='top', fontsize=8.5,
            color='#7722AA', bbox=bbox)

    # Title includes disrupted nodes
    dis     = list(res['initial_disruption'].keys())
    dis_str = ', '.join(dis[:2]) + ('...' if len(dis) > 2 else '')
    ax.set_title(f'{label}\n({dis_str})', color='#333333', fontsize=10, pad=7)
    ax.set_xlabel('Actual Severity',    fontsize=9)
    ax.set_ylabel('Predicted Severity', fontsize=9)
    ax.set_xlim(-0.03, 1.06)
    ax.set_ylim(-0.03, 1.06)
    ax.set_facecolor(BG)
    ax.grid(True, alpha=0.6)

# Shared legend (on third axis)
lh = [mpatches.Patch(color=LAYER_COLORS[l], label=l.capitalize()) for l in LAYER_ORDER]
lh += [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='gray',
           markersize=8, linestyle='None', label='GNN prediction (●)'),
    Line2D([0],[0], marker='X', color='gray',
           markersize=8, linestyle='None', label='Baseline prediction (✕)'),
    Line2D([0],[0], color='#AAAAAA', lw=1.2, linestyle='--', label='Perfect prediction'),
]
axes[2].legend(handles=lh, loc='lower right', fontsize=8, framealpha=0.85)

fig.suptitle('Predicted vs Actual Disruption Severity — Three Scenarios',
             color='#333333', fontsize=14, y=1.02)
plt.tight_layout()
p = RESULTS / f'dtnet_scenario_analysis_{DATE}.png'
fig.savefig(p, dpi=200, bbox_inches='tight', facecolor=BG)
print(f'Saved: {p}')
plt.show()

## 7. Summary & Conclusions

Quantitative comparison of DTNetGNN vs IsolatedBaseline and interpretation of the results.

In [ ]:
gm = eval_results['gnn_test']
bm = eval_results['baseline_test']

mse_red = (bm['mse'] - gm['mse']) / (bm['mse'] + 1e-10) * 100
mae_red = (bm['mae'] - gm['mae']) / (bm['mae'] + 1e-10) * 100
r2_gain = gm['r2'] - bm['r2']

sep = '=' * 60
print(sep)
print('  DTNet — GNN vs Isolated Baseline  (Test Set)')
print(sep)
print(f'  {"Metric":<8}  {"GNN":>12}  {"Baseline":>12}  {"Delta":>10}')
print(f'  {"-"*8}  {"-"*12}  {"-"*12}  {"-"*10}')
print(f'  {"MSE":<8}  {gm["mse"]:>12.6f}  {bm["mse"]:>12.6f}  {-mse_red:>+9.1f}%')
print(f'  {"MAE":<8}  {gm["mae"]:>12.6f}  {bm["mae"]:>12.6f}  {-mae_red:>+9.1f}%')
print(f'  {"R2":<8}  {gm["r2"]:>12.4f}  {bm["r2"]:>12.4f}  {r2_gain:>+10.4f}')
print()

if gm['r2'] > bm['r2']:
    print('  CONCLUSION: GNN outperforms the isolated baseline.')
    print(f'  MSE reduced by {mse_red:.1f}%  — graph structure improves cascade prediction.')
    print(f'  MAE reduced by {mae_red:.1f}%  — per-node severity estimates are more accurate.')
    print(f'  R2 gain of {r2_gain:.4f} — GNN explains more variance in disruption outcomes.')
else:
    print('  NOTE: Baseline matches or exceeds GNN. Check training convergence.')

print()
print('  Per-node-type breakdown (GNN R2 vs Baseline R2):')
print(f'  {"Layer":<14}  {"GNN R2":>8}  {"Base R2":>8}  {"Delta":>8}  {"Winner":>8}')
print(f'  {"-"*14}  {"-"*8}  {"-"*8}  {"-"*8}  {"-"*8}')
for layer in LAYER_ORDER:
    gm2 = eval_results['per_type_gnn'].get(layer)
    bm2 = eval_results['per_type_base'].get(layer)
    if gm2 and bm2:
        delta  = gm2['r2'] - bm2['r2']
        winner = 'GNN' if delta >= 0 else 'Base'
        print(f'  {layer:<14}  {gm2["r2"]:>8.3f}  {bm2["r2"]:>8.3f}  {delta:>+8.3f}  {winner:>8}')
print(sep)
print()
print('Figures saved:')
for fname in sorted(RESULTS.glob('dtnet_*.png')):
    print(f'  {fname}')